# FFAI RAG Pipeline: End-to-End with Real LLM

This notebook demonstrates the full RAG pipeline using **real LLM calls**
through `FFAI.query()`. No mocks.

Pipeline: `Document → Chunk → Index (BM25) → Search → FFAI.query() → Answer`

Requires a `MISTRAL_API_KEY` (or other provider key) in your environment.

## Section 1: Setup

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from src.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from src.FFAI import FFAI  # noqa: E402
from src.rag import RAG  # noqa: E402

print(f'Project root: {_project_root}')
print('Imports successful')

<div class="page-break"></div>

---

## Section 2: Create LLM Client and RAG

`RAG(bm25_alpha=0.6)` enables BM25 keyword search without requiring ChromaDB.
The `FFAI` wrapper wires the client to RAG via `ClientAdapter` internally.

In [ ]:
client = FFLiteLLMClient(
    model_string='mistral/mistral-small-2503',
    temperature=0.3,
    max_tokens=512,
    system_instructions='You are a helpful assistant. Answer concisely based on the provided context.',
)

rag = RAG(
    embed='mistral/mistral-embed',
    bm25_alpha=0.6,
    chunker='markdown',
    chunk_size=500,
    chunk_overlap=50,
)

ffai = FFAI(client=client, rag=rag)

print(f'Client: {client}')
print(f'RAG: bm25_alpha={rag._bm25_alpha}, store={rag._store is not None}')
print(f'FFAI ready: rag={ffai.rag is not None}')

<div class="page-break"></div>

---

## Section 3: Load and Index Documents

Download sample documents and index them into BM25. Each call to `rag.index()`
chunks the text and adds it to the BM25 index.

In [ ]:
from examples._rag_data.download import download

docs = download()
for name, path in docs.items():
    print(f'  {name}: {path.stat().st_size:,} bytes')

In [ ]:
python_tutorial = docs['python_tutorial.md'].read_text(encoding='utf-8')
api_docs = docs['api_docs.md'].read_text(encoding='utf-8')

n1 = rag.index(python_tutorial, source='python_tutorial.md')
n2 = rag.index(api_docs, source='api_docs.md')

print(f'Indexed {n1} chunks from python_tutorial.md')
print(f'Indexed {n2} chunks from api_docs.md')
print(f'Total chunks in BM25: {rag.count()}')

<div class="page-break"></div>

---

## Section 4: Inspect Chunks

Preview what the chunker produced.

In [ ]:
chunks = rag.chunk(python_tutorial, source='python_tutorial.md')

print(f'Total chunks: {len(chunks)}')
print()
for i, chunk in enumerate(chunks[:3]):
    preview = chunk.content[:100].replace(chr(10), ' ')
    print(f'  Chunk {i}: chars={chunk.start_char}-{chunk.end_char} | {preview}...')

<div class="page-break"></div>

---

## Section 5: Search (Retrieval Only)

Use `rag.search()` to retrieve relevant chunks without generating an answer.

In [ ]:
hits = rag.search('async programming', top_k=3)

print(f'Top {len(hits)} results for "async programming":')
print()
for i, hit in enumerate(hits):
    preview = hit.content[:120].replace(chr(10), ' ')
    print(f'  [{i+1}] score={hit.score:.4f} source={hit.source}')
    print(f'      {preview}...')
    print()

<div class="page-break"></div>

---

## Section 6: End-to-End with `FFAI.query()`

One call: search → format context → prompt → LLM → answer.
`FFAI.query()` internally creates a `ClientAdapter` that bridges the LLM client.

In [ ]:
result = ffai.query(
    'How do I run multiple async tasks concurrently in Python?',
    top_k=3,
)

print(f'Answer:\n{result.answer}')
print(f'\nSources: {result.sources}')
print(f'Hits: {len(result.hits)}')
print(f'Prompt length: {len(result.prompt):,} chars')

<div class="page-break"></div>

---

## Section 7: Multiple Queries

Run several questions to show the RAG pipeline across different topics.

In [ ]:
questions = [
    'What is the purpose of asyncio.gather()?',
    'How do I handle timeouts in async code?',
    'What are the rate limits for the login endpoint?',
    'What error format does the API use?',
]

for q in questions:
    result = ffai.query(q, top_k=3)
    answer_preview = result.answer[:150].replace(chr(10), ' ')
    print(f'Q: {q}')
    print(f'A: {answer_preview}...')
    print(f'   Sources: {result.sources}')
    print()

<div class="page-break"></div>

---

## Section 8: Custom Prompt Template

Override the default prompt template for domain-specific formatting.

In [ ]:
custom_template = (
    'You are a Python expert. Using ONLY the context below, answer the question.\n\n'
    'Context:\n{context}\n\n'
    'Question: {question}\n\n'
    'Provide a code example if applicable.'
)

result = ffai.query(
    'How do task groups work?',
    top_k=3,
    prompt_template=custom_template,
)

print(f'Answer:\n{result.answer}')
print(f'\nSources: {result.sources}')

<div class="page-break"></div>

---

## Section 9: Inspect the Generated Prompt

See exactly what was sent to the LLM — context + question.

In [ ]:
result = ffai.query('What are async context managers?', top_k=2)

print('=== Full Prompt Sent to LLM ===')
print(result.prompt)
print('=== End Prompt ===')
print(f'\nAnswer: {result.answer[:200]}...')

<div class="page-break"></div>

---

## Section 10: Summary

In [ ]:
import polars as pl

summary = pl.DataFrame([
    {'Step': 1, 'Component': 'Client', 'API': 'FFLiteLLMClient(model_string=...)', 'Role': 'LLM provider'},
    {'Step': 2, 'Component': 'RAG', 'API': 'RAG(bm25_alpha=0.6)', 'Role': 'Chunk + index + retrieve'},
    {'Step': 3, 'Component': 'FFAI', 'API': 'FFAI(client, rag=rag)', 'Role': 'Wire client to RAG'},
    {'Step': 4, 'Component': 'Index', 'API': 'rag.index(text, source=...)', 'Role': 'Chunk & add to BM25'},
    {'Step': 5, 'Component': 'Query', 'API': 'ffai.query(question, top_k=3)', 'Role': 'Search → format → generate'},
])

print(summary)